# InfiniteTalk｜G4高速版 v7｜音声の長さを自動検出して動画生成

**前回40秒の動画生成に成功したv6をベースにした高速版。** 画像1枚と音声1つをアップロードすると、音声の長さを測定して自動的に生成尺を決めます。**40秒固定ではありません。**

### 新しい点
- **G4高速（初期値）**：GPU VRAMが十分なとき、CPUオフロードを無効化。G4 (96GB) を想定
- **安定版（v6同等）／省VRAM**にワンクリックで切替可能。量子化済み8ステップモデルはそのまま
- 音声の実時間（例：37.46秒、65.18秒）を `ffprobe` で検出。**短縮も40秒への無音延長もしません**
- 例外：モデルが最初に81フレームを要求するため、約4秒未満の音声のみ内部で無音を足し、最終MP4は元の長さへ戻す
- **短時間テスト**：設定セルで「先頭N秒だけ使う」を選択可能（初期値0 = 無制限）
- 生成ログ、GPU診断、推定区間数、出力再取得。**⑦に失敗しても生成結果は保持**
- TeaCacheは試験オプション（画質・時間への影響は環境次第）。**FlashAttention 2は不要**：成功済みSDPA代替を保持

### 実行方法
初回は **① → ② → ②-B → ③ → ④ → ⑤ → ⑥ → ⑦**。同じColabセッションで2回目からは **④ → ⑤ → ⑥ → ⑦**。すでにv6が動くセッションなら、**②と③の再ダウンロードは不要**（②-Bが未適用なら実行）。

**重要：** 高速化倍率は未検証です。オフロードを切るとGPUメモリ消費が増えます。初回は「先頭N秒だけ使う」を5にして試し、良ければ0に戻してください。長尺はColabの実行時間制限やディスク・RAM容量に注意してください。

[InfiniteTalk公式](https://github.com/MeiGen-AI/InfiniteTalk) / [量子化モデル](https://huggingface.co/MeiGen-AI/InfiniteTalk/tree/main/quant_models)


## ① GPU・空き容量を確認

In [ ]:
import subprocess, shutil, re, os, sys

def cmd_out(cmd):
    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT).strip()

try:
    gpus = cmd_out(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'])
except Exception as e:
    raise RuntimeError('GPUが有効ではありません。Colabの「ランタイムのタイプを変更」でG4またはA100を選んでください。') from e
print('GPU:', gpus)
mem_mib = int(gpus.splitlines()[0].rsplit(',', 1)[-1].strip())
mem_gib = mem_mib / 1024
free_gib = shutil.disk_usage('/content').free / (1024**3)
print(f'VRAM: {mem_gib:.1f} GiB / ディスク空き: {free_gib:.1f} GiB')
print('Python:', sys.version.split()[0])
if mem_gib < 38:
    raise RuntimeError('この14Bモデルの40秒生成にはVRAMが不足しやすいです。G4またはA100を選んでください。')
if free_gib < 42:
    raise RuntimeError('モデル取得に必要なディスク空きが足りません。45〜55GiB程度の空きを確保してください。')
if mem_gib < 55:
    print('⚠ 40GiB前後のGPU：モデルをCPUへ退避する低VRAMモードを自動使用します。速度が低下します。')
print('✅ 事前チェック完了')

## ② Colab環境と分離したPython 3.10を構築（初回のみ）

**ColabのPython 3.13は変更しません。** `uv`を使ってPython 3.10仮想環境を作り、InfiniteTalk専用に必要なライブラリをインストールします。

エラー時は下部のインストールログに実際の原因が表示されます。**古いノートブックで途中失敗していても、そのままこのセルを実行できます。** これまでのコード取得・モデルダウンロードが残っていれば再利用します。


In [ ]:
import os, sys, subprocess, pathlib, shutil, textwrap
ROOT = pathlib.Path('/content/InfiniteTalk')
VENV = pathlib.Path('/content/infinitetalk_py310')
VENV_PY = VENV / 'bin' / 'python'

# ログを省略しない。パッケージ名・依存エラーが見えるようにする。
def run(command, *, cwd=None):
    command = list(map(str, command))
    print('▶', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

# 初期化: 以前のColab Python 3.13向けpip失敗は無視して独立環境を使う。
os.environ['PIP_NO_CACHE_DIR'] = '1'
if not (ROOT / 'generate_infinitetalk.py').is_file():
    run(['git','clone','--depth','1','https://github.com/MeiGen-AI/InfiniteTalk.git',ROOT])
else:
    print('✅ 既存のInfiniteTalkリポジトリを再利用')

# uv はColabカーネルの Python 3.13 にインストールしても、モデル環境を汚さない。
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'uv'])
UV = shutil.which('uv')
assert UV, 'uvのインストールを確認できません。'

if not VENV_PY.exists():
    run([UV, 'python', 'install', '3.10'])
    run([UV, 'venv', '--python', '3.10', str(VENV)])
print('専用Python:', subprocess.check_output([str(VENV_PY), '-V'], text=True).strip())

# PyTorch/xFormersのABI/CUDA互換性を揃える（G4 Blackwell向けCUDA 12.8）。
run([UV, 'pip', 'install', '--python', str(VENV_PY),
     '--index-url', 'https://download.pytorch.org/whl/cu128',
     'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1'])

# ColabではGradio UIやDashscope APIを使わないため取り除き、依存関係の将来変更を回避。
# xfuserは単一GPUでも一部の公式コードからimportされるので2025年互換版を維持。
REQ = ROOT / 'requirements.colab.txt'
REQ.write_text('\n'.join(line for line in (ROOT / 'requirements.txt').read_text().splitlines()
                       if not line.strip().startswith(('gradio', 'dashscope', 'xfuser'))) + '\n')
# torchの版を固定し、インストール時のGPUパッケージ入れ替えを防ぐ。
run([UV, 'pip', 'install', '--python', str(VENV_PY),
     '-r', str(REQ), 'xfuser==0.4.4',
     'torch==2.7.1', 'xformers==0.0.31.post1',
     'numpy==1.26.4', 'opencv-python==4.10.0.84',
     'transformers==4.49.0', 'diffusers==0.33.1',
     'huggingface_hub==0.34.4', 'librosa==0.10.2.post1',
     'misaki[en]', 'soundfile', 'hf-xet', 'ninja'])

# FP8 T5をGPUに転送する際、optimum-quantoがC++/CUDA拡張をJITビルドする。
# uvが作った仮想環境にはpipが入っていない場合があるため、必ずuv pipでインストールする。
# venv/binを PATH の先頭に追加し、torch.utils.cpp_extension がninjaを見つけられるようにする。
os.environ['PATH'] = str(VENV / 'bin') + os.pathsep + os.environ.get('PATH', '')
run(['ninja', '--version'])
print('✅ Ninja を発見。CUDA拡張のビルドに利用できます。')
# nvccがない場合は、次のGPU生成セルでエラーになる可能性がある。ここで状況を可視化。
if shutil.which('nvcc'):
    result = subprocess.run(['nvcc', '--version'], text=True, capture_output=True)
    print('CUDA compiler:', (result.stdout.strip().splitlines() or ['unknown'])[-1])
else:
    print('⚠️ nvccが PATH に見つかりません。次にCUDA toolkit関連のエラーが出る可能性があります。')

if shutil.which('ffmpeg') is None:
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','-qq','ffmpeg'])

# Pythonのバージョン変更に対応する公式ソースの最小限の互換性パッチ。
# パッチは繰り返し実行しても変化しない（冪等）。
mt = ROOT / 'wan' / 'multitalk.py'
s = mt.read_text()
if 'from inspect import ArgSpec' in s:
    s = s.replace('from inspect import ArgSpec', 'from inspect import FullArgSpec as ArgSpec')
    mt.write_text(s)
    print('✅ inspect.ArgSpec互換パッチを適用')
wv = ROOT / 'src' / 'audio_analysis' / 'wav2vec2.py'
s = wv.read_text()
needle = '        self.config.output_attentions = True'
if needle in s and '_attn_implementation = "eager"' not in s:
    s = s.replace(needle, '        self.config._attn_implementation = "eager"\n' + needle)
    wv.write_text(s)
    print('✅ Wav2Vec2 eager attention互換パッチを適用')

# エラーを早めに検出。OSカーネルではなく専用Pythonで実行。
run([str(VENV_PY), '-c',
     'import torch, torchvision, xformers, transformers, diffusers, numpy; '
     'import wan.multitalk; '
     'print("環境OK: torch", torch.__version__, "cuda", torch.version.cuda, "numpy", numpy.__version__)'],
    cwd=ROOT)
print('✅ 環境構築完了。Colabランタイムの再起動は不要です。')


## ②-B FlashAttentionを必要としないGPU互換パッチ（初回のみ）

**②の直後に実行してください。** 既存のv4でモデルを取得済みの場合、下のパッチセルだけ今のColabに追加して実行し、⑤→⑥→⑦を再実行すればOKです。

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path('/content/InfiniteTalk')
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
attn_file = ROOT / 'wan' / 'modules' / 'attention.py'
assert attn_file.is_file(), '②でInfiniteTalkをセットアップしてください。'
assert VENV_PY.is_file(), '②でPython 3.10環境を構築してください。'

s = attn_file.read_text(encoding='utf-8')
marker = '# COLAB_SDPA_FALLBACK_V5'
needle = '    half_dtypes = (torch.float16, torch.bfloat16)'

fallback = '''    # COLAB_SDPA_FALLBACK_V5
    # FlashAttention 2 is unavailable in some Colab runtimes (including G4).
    # Use PyTorch SDPA while preserving the [B, seq, heads, dim] API and dtype.
    if not FLASH_ATTN_2_AVAILABLE and (version == 2 or not FLASH_ATTN_3_AVAILABLE):
        b, lq, n, d = q.shape
        lk = k.shape[1]
        out_dtype = q.dtype
        work_dtype = v.dtype if v.dtype in (torch.float16, torch.bfloat16) else dtype
        qq = q.transpose(1, 2).to(dtype=work_dtype)
        kk = k.transpose(1, 2).to(dtype=work_dtype)
        vv = v.transpose(1, 2).to(dtype=work_dtype)
        if q_scale is not None:
            qq = qq * q_scale

        mask = None
        if k_lens is not None:
            kl = torch.as_tensor(k_lens, device=q.device).reshape(b)
            valid_keys = torch.arange(lk, device=q.device)[None, None, None, :] < kl[:, None, None, None]
            mask = valid_keys
        if window_size is not None and tuple(window_size) != (-1, -1):
            left, right = window_size
            qi = torch.arange(lq, device=q.device)[None, None, :, None]
            ki = torch.arange(lk, device=q.device)[None, None, None, :]
            local = torch.ones((1, 1, lq, lk), device=q.device, dtype=torch.bool)
            if left >= 0:
                local = local & (ki >= qi - left)
            if right >= 0:
                local = local & (ki <= qi + right)
            mask = local if mask is None else mask & local
        # Build an explicit causal mask when necessary to combine with key masking.
        if causal:
            qi = torch.arange(lq, device=q.device)[None, None, :, None]
            ki = torch.arange(lk, device=q.device)[None, None, None, :]
            # FlashAttention varlen causal alignment is bottom-right for Lq != Lk.
            causal_mask = ki <= qi + (lk - lq)
            mask = causal_mask if mask is None else mask & causal_mask

        result = torch.nn.functional.scaled_dot_product_attention(
            qq, kk, vv, attn_mask=mask, dropout_p=dropout_p,
            is_causal=False, scale=softmax_scale)
        if q_lens is not None:
            ql = torch.as_tensor(q_lens, device=q.device).reshape(b)
            valid_q = torch.arange(lq, device=q.device)[None, None, :, None] < ql[:, None, None, None]
            result = result.masked_fill(~valid_q, 0)
        return result.transpose(1, 2).contiguous().to(out_dtype)

'''

if marker in s:
    print('✅ SDPAパッチは既に適用済み（再実行しても問題ありません）')
else:
    assert needle in s, 'attention.pyのコード構造が変わっています。パッチ適用を中止しました。'
    s = s.replace(needle, fallback + needle, 1)
    # Syntax validation before modifying checked-out source
    compile(s, str(attn_file), 'exec')
    attn_file.with_suffix('.py.before_colab_sdpa_v5').write_text(attn_file.read_text(encoding='utf-8'), encoding='utf-8')
    attn_file.write_text(s, encoding='utf-8')
    print('✅ FlashAttention不在時のPyTorch SDPAフォールバックを追加')

# New subprocess: no runtime restart. Two tests including key padding and causal behavior.
probe = r'''
import torch
from wan.modules.attention import flash_attention
assert torch.cuda.is_available(), 'GPUが有効ではありません'
dev = 'cuda'
for lq, lk, causal in [(8, 8, False), (5, 9, True)]:
    q = torch.randn(2, lq, 4, 32, device=dev, dtype=torch.float16)
    k = torch.randn(2, lk, 4, 32, device=dev, dtype=torch.float16)
    v = torch.randn(2, lk, 4, 32, device=dev, dtype=torch.float16)
    k_lens = torch.tensor([lk, lk - 1], device=dev, dtype=torch.int32)
    out = flash_attention(q, k, v, k_lens=k_lens, causal=causal, version=2)
    assert out.shape == q.shape and out.dtype == q.dtype
    assert torch.isfinite(out).all()
print('✅ Attention動作テスト成功 / GPU:', torch.cuda.get_device_name(0))
'''
subprocess.run([str(VENV_PY), '-c', probe], cwd=str(ROOT), check=True)
print('✅ このあと⑤で画像と音声を用意し、⑥→⑦を実行してください。')


## ③ 量子化モデルをダウンロード（初回のみ）

ダウンロードには **②で構築した専用Python** を使います。以前のダウンロードが残っていれば再利用します。必要なベースモデル付属ファイル・量子化DiT・量子化T5・Wav2Vec2のみを取得します（数十GB規模）。

途中切断はこのセルを再実行して再開。Hugging Face上のモデルが削除・非公開化されている場合はエラーとして停止します。


In [ ]:
from pathlib import Path
import os, subprocess, shutil, textwrap
ROOT = Path('/content/InfiniteTalk')
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
assert VENV_PY.exists(), '②の環境構築から実行してください。'
W = ROOT / 'weights'
BASE = W / 'Wan2.1-I2V-14B-480P'
IT = W / 'InfiniteTalk'
WV = W / 'chinese-wav2vec2-base'
QUANT_NAME = 'infinitetalk_single_int8_lora'
QUANT = IT / 'quant_models' / (QUANT_NAME + '.safetensors')

free = shutil.disk_usage('/content').free / 1024**3
if not QUANT.exists() and free < 32:
    raise RuntimeError(f'ディスク空き {free:.1f} GiB。容量不足の可能性。')

# HFライブラリも仮想環境側。ノートブックカーネルへインポートしない。
download_script = r'''
from pathlib import Path
from huggingface_hub import snapshot_download, hf_hub_download
W = Path('/content/InfiniteTalk/weights')
BASE = W / 'Wan2.1-I2V-14B-480P'
IT = W / 'InfiniteTalk'
WV = W / 'chinese-wav2vec2-base'
print('1/3: Wan2.1補助モデル', flush=True)
snapshot_download('Wan-AI/Wan2.1-I2V-14B-480P',local_dir=str(BASE),
  allow_patterns=['config.json','Wan2.1_VAE.pth',
   'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
   'google/**','xlm-roberta-large/**'],max_workers=3)
print('2/3: InfiniteTalk INT8+LoRA / FP8 T5', flush=True)
snapshot_download('MeiGen-AI/InfiniteTalk',local_dir=str(IT),
  allow_patterns=['quant_models/infinitetalk_single_int8_lora.safetensors',
   'quant_models/infinitetalk_single_int8_lora.json',
   'quant_models/t5_fp8.safetensors','quant_models/t5_map_fp8.json'],max_workers=3)
print('3/3: 音声エンコーダー', flush=True)
snapshot_download('TencentGameMate/chinese-wav2vec2-base',local_dir=str(WV),
  allow_patterns=['*.json','*.txt','*.model'], max_workers=3)
hf_hub_download('TencentGameMate/chinese-wav2vec2-base', filename='model.safetensors',
  revision='refs/pr/1',local_dir=str(WV))
print('✅ ダウンロード完了', flush=True)
'''
subprocess.run([str(VENV_PY), '-c', download_script], cwd=str(ROOT), check=True)

needed = [BASE/'config.json', BASE/'Wan2.1_VAE.pth',
 BASE/'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
 QUANT, IT/'quant_models'/'infinitetalk_single_int8_lora.json',
 IT/'quant_models'/'t5_fp8.safetensors', IT/'quant_models'/'t5_map_fp8.json',
 WV/'model.safetensors']
missing = [str(p) for p in needed if not p.is_file()]
assert not missing, '必須ファイル不足（③を再実行）:\n' + '\n'.join(missing)
for p in needed:
    print(f'✅ {p.relative_to(ROOT)} ({p.stat().st_size / 1024**2:,.1f} MiB)')


## ④ 生成設定（通常はここから）

**初期値：「G4高速」「音声全体を使用」「480p/8ステップ」**。音声の長さは⑤のアップロード後に自動で決まります。

- `PERFORMANCE_MODE`：G4高速（CPUオフロード無効）／安定版（v6と同じ）／省VRAM
- `TEST_FIRST_SECONDS`：初回の確認用。`5`なら音声の最初の5秒だけ生成。`0`なら音声の全長を使用
- `ENABLE_TEACACHE`：追加の高速化を試す任意設定。量子化8ステップでの効果・画質は未検証なので初期値はOFF

CPUオフロード無効化はG4やA100 80GBなど大容量GPU向け。VRAMが少ないGPUでは安全なモードへ自動切替します。


In [ ]:
# @title 生成オプション（音声の長さは自動検出）
PERFORMANCE_MODE = "G4高速"  # @param ["G4高速", "安定版", "省VRAM"]
TEST_FIRST_SECONDS = 0.0  # @param {type:"number"}
ENABLE_TEACACHE = False  # @param {type:"boolean"}
TEACACHE_THRESHOLD = 0.15  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
PROMPT = "A single anime-style young adult woman speaks naturally to the camera, accurate lip synchronization with her speech, expressive facial movements, subtle natural head movements, gentle upper-body movement and occasional small hand gestures. Keep the character identity, hair, outfit, camera angle and background consistent. No scene cuts, no extra people."  # @param {type:"string"}
AUTO_DOWNLOAD = True  # @param {type:"boolean"}

assert PERFORMANCE_MODE in ("G4高速", "安定版", "省VRAM")
assert 0 <= float(TEST_FIRST_SECONDS) <= 3600, "テスト制限は0〜3600秒。0なら音声の全長を使用します。"
assert 0 < float(TEACACHE_THRESHOLD) <= 1.0
assert isinstance(SEED, int)
print(f"✅ モード: {PERFORMANCE_MODE} / 音声長: 自動 / テスト上限: {TEST_FIRST_SECONDS}秒（0は無制限）")
print(f"✅ 480p・8ステップ / TeaCache: {'ON（試験的）' if ENABLE_TEACACHE else 'OFF'}")


## ⑤ 画像＋音声をアップロード（音声長を自動検出）

画像を1つ、音声を1つ選択。音声を16kHzモノラルWAVに変換して正確な秒数を測り、`TARGET_SEC` に保存します。

**40秒に揃えたり、無音を追加したり、勝手に切り取ったりしません。** `TEST_FIRST_SECONDS > 0` の場合だけ意図的に先頭を切ります。モデル内部の最低長さだけ例外です。

長時間（例：10分）の音声はそのまま使用できますが、生成時間とColab利用制限に注意。音声の長さと推定区間数を⑥の前に表示します。


In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from pathlib import Path
import subprocess, json, datetime, uuid, shutil, math, wave

assert 'TEST_FIRST_SECONDS' in globals(), '先に④生成設定を実行してください。'
print('画像1つ＋音声1つを選択してください（例：PNG/JPGとWAV/MP3/M4A）')
uploaded = files.upload()
image_ext = {'.png','.jpg','.jpeg','.webp','.bmp'}
audio_ext = {'.wav','.mp3','.m4a','.aac','.flac','.ogg','.opus','.wma','.mp4'}
imgs = [Path(x) for x in uploaded if Path(x).suffix.lower() in image_ext]
auds = [Path(x) for x in uploaded if Path(x).suffix.lower() in audio_ext]
assert len(imgs) == 1 and len(auds) == 1, (
    f'画像1個と音声1個を選んでください（画像{len(imgs)}個・音声{len(auds)}個）')

job_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
JOB = Path('/content/infinitetalk_outputs') / job_id
JOB.mkdir(parents=True, exist_ok=True)
IMAGE_PATH = JOB / 'reference.png'
AUDIO_PATH = JOB / 'speech_16k_mono.wav'

with Image.open(imgs[0]) as im:
    im = ImageOps.exif_transpose(im).convert('RGB')
    im.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    im.save(IMAGE_PATH, optimize=True)

# 長さはffprobeメタデータではなく、16kHz PCMへ変換した後のサンプル数で測定。
# 0: 全長。テスト上限がある場合だけ意図的に先頭 N 秒を使用する。
ff = ['ffmpeg','-y','-nostdin','-hide_banner','-loglevel','error',
      '-i', str(auds[0]), '-vn', '-ac','1','-ar','16000','-c:a','pcm_s16le']
if TEST_FIRST_SECONDS > 0:
    ff += ['-t', str(float(TEST_FIRST_SECONDS))]
ff += [str(AUDIO_PATH)]
subprocess.run(ff, check=True)

with wave.open(str(AUDIO_PATH), 'rb') as wf:
    assert wf.getframerate() == 16000 and wf.getnchannels() == 1
    TARGET_SEC = wf.getnframes() / wf.getframerate()
assert TARGET_SEC >= 0.5, f'有効な音声が短すぎます（{TARGET_SEC:.2f}秒）。0.5秒以上を用意してください。'

# モデルは81フレームより長い音声埋め込みを要求する。
# 4秒未満のみ内部入力を4.0秒までpadding。MP4仕上げは元のTARGET_SECで切り戻す。
MIN_MODEL_SEC = 4.0
if TARGET_SEC < MIN_MODEL_SEC:
    padded = JOB / 'speech_16k_model.wav'
    subprocess.run(['ffmpeg','-y','-nostdin','-hide_banner','-loglevel','error',
                    '-i', str(AUDIO_PATH), '-af', f'apad=whole_dur={MIN_MODEL_SEC}',
                    '-t', str(MIN_MODEL_SEC), '-ac','1','-ar','16000',
                    '-c:a','pcm_s16le', str(padded)], check=True)
    MODEL_AUDIO_PATH = padded
    print('ℹ️ 短い音声のため、モデル内部だけ4秒に無音延長。完成MP4は元の長さに戻します。')
else:
    MODEL_AUDIO_PATH = AUDIO_PATH
MODEL_SEC = max(TARGET_SEC, MIN_MODEL_SEC)
FRAME_COUNT = math.ceil(MODEL_SEC * 25)
# InfiniteTalkは最初81フレーム、後続は72フレームずつ増える。
CHUNK_COUNT = max(1, 1 + math.ceil(max(0, FRAME_COUNT - 81) / 72))

JSON_PATH = JOB / 'input.json'
JSON_PATH.write_text(json.dumps({
    'prompt': PROMPT, 'cond_video': str(IMAGE_PATH),
    'cond_audio': {'person1': str(MODEL_AUDIO_PATH)},
}, ensure_ascii=False, indent=2), encoding='utf-8')

META_PATH = JOB / 'job_meta.json'
META_PATH.write_text(json.dumps({
    'target_seconds': TARGET_SEC,
    'model_seconds': MODEL_SEC,
    'frame_count': FRAME_COUNT,
    'chunk_count_est': CHUNK_COUNT,
    'performance_mode': PERFORMANCE_MODE,
    'enable_teacache': ENABLE_TEACACHE,
    'test_limit_seconds': TEST_FIRST_SECONDS,
    'input_filename_audio': auds[0].name,
}, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'✅ 音声の長さ: {TARGET_SEC:.2f}秒 → 動画の目標長さも {TARGET_SEC:.2f}秒')
print(f'✅ 内部生成: {FRAME_COUNT}フレーム（約{CHUNK_COUNT}区間）')
print(f'✅ JOB: {JOB}')
if TARGET_SEC > 90:
    print('⚠ 長尺です。Colabランタイム切断・RAM不足や長時間の生成に注意。短時間テストを推奨します。')


### ⑤-補足　生成前のセルフチェック（必要なときに実行）

生成前にGPUメモリ・ディスク・必要ファイルの有無を確認します。失敗原因の切り分け用。パスやサイズ以外の個人データは表示しません。

In [ ]:
from pathlib import Path
import subprocess, shutil
ROOT = Path('/content/InfiniteTalk')
BASE = ROOT / 'weights' / 'Wan2.1-I2V-14B-480P'
IT = ROOT / 'weights' / 'InfiniteTalk'
WV = ROOT / 'weights' / 'chinese-wav2vec2-base'
required = {
    'Wan config': BASE / 'config.json',
    'Wan VAE': BASE / 'Wan2.1_VAE.pth',
    'Wan CLIP': BASE / 'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
    'INT8+LoRA DiT': IT / 'quant_models' / 'infinitetalk_single_int8_lora.safetensors',
    'INT8+LoRA map': IT / 'quant_models' / 'infinitetalk_single_int8_lora.json',
    'FP8 T5': IT / 'quant_models' / 't5_fp8.safetensors',
    'FP8 T5 map': IT / 'quant_models' / 't5_map_fp8.json',
    'Audio encoder': WV / 'model.safetensors',
}
print('== GPU ==')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],check=False)
print('== ディスク ==')
print(f"{shutil.disk_usage('/content').free / 1024**3:.1f} GiB 空き")
print('== チェックポイント ==')
failed=[]
for key,path in required.items():
    ok = path.is_file() and path.stat().st_size > 0
    label = f'{path.stat().st_size / 1024**3:.2f} GiB' if ok else '見つからない / サイズ0'
    print(('✅' if ok else '❌'),key,':',label)
    if not ok: failed.append(str(path))
if failed:
    print('\nモデルファイルが不足しています。③のセルを再実行してください。')
else:
    print('\n✅ 基本ファイルは揃っています。')

## ⑥ InfiniteTalkを生成（G4高速 / 安定版 切替・実ログ表示）

⑤で決まった音声長に合わせて `--max_frame_num` を設定。**8ステップ量子化モデルを維持**し、G4高速モードでは `--offload_model False` でCPU/GPU間の退避を減らします。低VRAMでは自動的に安定版へ切替。

TeaCacheは④で任意に有効化できますが、品質や効果は未検証。**4ステップ化やFlashAttention 2の強制インストールはしません。** 初回は④のテスト上限を5にし、短尺で成功と速度を確認してください。

このセルが失敗しても理由をログから表示し、出力フォルダは消しません。CUDA OOM時も**勝手に高コストの再生成を開始しません**。④を「安定版」に変更して⑤からやり直してください。


In [ ]:
from pathlib import Path
import subprocess, os, json, shutil, math
# 既存のColabランタイムを再利用した場合もモデルパスを再構成する。
ROOT = Path('/content/InfiniteTalk')
BASE = ROOT / 'weights' / 'Wan2.1-I2V-14B-480P'
WV = ROOT / 'weights' / 'chinese-wav2vec2-base'
QUANT = ROOT / 'weights' / 'InfiniteTalk' / 'quant_models' / 'infinitetalk_single_int8_lora.safetensors'
assert Path(JSON_PATH).exists(), '先に⑤アップロードを実行してください。'
assert QUANT.exists(), '先に③モデル取得を実行してください。'
assert FRAME_COUNT >= 81 and TARGET_SEC >= 0.5
RAW_PREFIX = JOB / 'infinitetalk_raw'
LOG_FILE = JOB / 'generate.log'
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
assert VENV_PY.exists(), '②環境構築を実行してください。'

# 実際のGPU空き容量をチェック。G4約96GBで余裕がある場合だけ高速モード。
try:
    gpu_csv = subprocess.check_output(
        ['nvidia-smi','--query-gpu=memory.total,memory.free','--format=csv,noheader,nounits'],
        text=True).splitlines()[0]
    total_mib, free_mib = [int(v.strip()) for v in gpu_csv.split(',')]
except Exception as e:
    raise RuntimeError('GPUが見つかりません。①でGPUを確認してください。') from e
vram_total_gib, vram_free_gib = total_mib / 1024, free_mib / 1024

requested_mode = PERFORMANCE_MODE
actual_mode = requested_mode
if requested_mode == 'G4高速' and (vram_total_gib < 70 or vram_free_gib < 65):
    actual_mode = '安定版'
    print(f'⚠ 空きVRAM {vram_free_gib:.1f} GiB：G4高速は危険なため安定版に切替。',flush=True)

offload = 'False' if actual_mode == 'G4高速' else 'True'
args = [str(VENV_PY), 'generate_infinitetalk.py',
        '--ckpt_dir', str(BASE),
        '--wav2vec_dir', str(WV),
        '--quant_dir', str(QUANT),
        '--quant', 'fp8',  # v6で実績のある量子化復元設定（INT8 DiTのマップはファイル側）
        '--input_json', str(JSON_PATH),
        '--size', 'infinitetalk-480',
        '--sample_steps', '8',
        '--sample_text_guide_scale', '1.0',
        '--sample_audio_guide_scale', '2.0',
        '--sample_shift', '2',
        '--mode', 'streaming',
        '--frame_num', '81',
        '--max_frame_num', str(int(FRAME_COUNT)),
        '--motion_frame', '9',
        '--base_seed', str(SEED),
        '--offload_model', offload,
        '--save_file', str(RAW_PREFIX),
        '--audio_save_dir', str(JOB/'audio_embeddings')]
if actual_mode == '省VRAM' or vram_total_gib < 55:
    args += ['--num_persistent_param_in_dit', '0', '--t5_cpu']
if ENABLE_TEACACHE:
    args += ['--use_teacache', '--teacache_thresh', str(float(TEACACHE_THRESHOLD))]

print(f'✅ 動画目標: {TARGET_SEC:.2f}秒 / {FRAME_COUNT}フレーム / 約{CHUNK_COUNT}区間',flush=True)
print(f'✅ モード: {actual_mode} / offload={offload} / TeaCache={ENABLE_TEACACHE}',flush=True)
print(f'✅ GPU空き: {vram_free_gib:.1f}/{vram_total_gib:.1f} GiB',flush=True)

# v3環境を同じColabセッションで使い回しても修復できるようにする。
# torch の拡張ロードでは Ninja が必要。uv venvにはpip自体がない場合もある。
import sys
VENV_BIN = VENV_PY.parent
if not (VENV_BIN / 'ninja').exists():
    uv_exe = shutil.which('uv')
    assert uv_exe, 'uvが見つかりません。先に②の環境構築を実行してください。'
    print('Ninjaがないため、専用環境に自動インストールします…', flush=True)
    subprocess.run([uv_exe, 'pip', 'install', '--python', str(VENV_PY), 'ninja'], check=True)

runtime_env = os.environ.copy()
runtime_env['PATH'] = str(VENV_BIN) + os.pathsep + runtime_env.get('PATH', '')
# ColabでGPU拡張の同時ビルド数が増えすぎないようにする。
runtime_env['MAX_JOBS'] = '2'
check_ninja = subprocess.run([str(VENV_BIN / 'ninja'), '--version'],
                             capture_output=True, text=True, env=runtime_env)
assert check_ninja.returncode == 0, f'Ninja が実行できません: {check_ninja.stderr}'
print('✅ Ninja:', check_ninja.stdout.strip(), flush=True)

runtime_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
runtime_env['TOKENIZERS_PARALLELISM'] = 'false'
print('生成開始（モデル読み込みと推論に時間がかかります）。出力:', RAW_PREFIX, flush=True)
with open(LOG_FILE, 'w', encoding='utf-8') as log:
    proc = subprocess.Popen(args, cwd=str(ROOT), env=runtime_env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        status = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        try: proc.wait(timeout=15)
        except subprocess.TimeoutExpired: proc.kill()
        raise
if status != 0:
    # 終了コードだけでは何が失敗したか分からない。常に元のエラーを表示する。
    import re
    lines = LOG_FILE.read_text(encoding='utf-8', errors='replace').splitlines()
    # ANSIカラーコードなどを除去し、エラー解析を読みやすくする。
    clean = [re.sub(r'\x1b\[[0-9;]*m', '', line) for line in lines]
    patterns = [r'Traceback', r'Error:', r'Exception:', r'CUDA out of memory',
                r'No such file', r'not found', r'unsupported', r'AssertionError',
                r'safetensors', r'Killed', r'undefined symbol', r'cannot import',
                r'Failed', r'RuntimeError', r'KeyError']
    interesting = [i for i, line in enumerate(clean)
                   if any(re.search(pat, line, re.I) for pat in patterns)]
    print('\n'+'='*24+' InfiniteTalk 失敗診断 '+'='*24)
    print('終了コード:', status)
    print('ログ:', LOG_FILE)
    if interesting:
        print('\n▼ 検出したエラー箇所（前後の文脈を含む）')
        a, b = max(0,interesting[-1]-24), min(len(clean),interesting[-1]+24)
        print('\n'.join(f'{i+1}: {clean[i]}' for i in range(a,b)))
    print('\n▼ ログ末尾（最大100行）')
    print('\n'.join(clean[-100:]))
    try:
        gpu_report = subprocess.run(['nvidia-smi','--query-gpu=name,memory.used,memory.free,memory.total',
                                     '--format=csv,noheader'], capture_output=True,text=True)
        print('\nGPU 状態:',gpu_report.stdout.strip() or gpu_report.stderr.strip())
    except Exception as ex:
        print('GPU診断不可:',ex)
    text_lower = '\n'.join(clean[-200:]).lower()
    if any(x in text_lower for x in ('out of memory','cuda error: out of memory', 'cuda oom')):
        print('\n推奨：GPUメモリ不足です。④を安定版か省VRAMに変え、⑤から再試行。')
    elif 'ninja is required' in text_lower:
        print('\n推奨：Ninjaが見つかりません。②を再実行するかv4ノートブックで再実行してください。')
    elif any(x in text_lower for x in ('nvcc', 'cuda_home', 'cuda toolkit')):
        print('\n推奨：CUDA拡張のビルドでエラー。ログ内のCUDA compiler / nvcc の記述を確認してください。')
    elif any(x in text_lower for x in ('no such file', 'filenotfounderror','entry not found')):
        print('\n推奨：モデルやファイルの欠損が考えられます。③のダウンロードを再実行してください。')
    elif any(x in text_lower for x in ('modulenotfounderror','importerror','undefined symbol')):
        print('\n推奨：ライブラリの互換性問題が考えられます。ログのエラー行を共有してください。')
    elif any(x in text_lower for x in ('no space left on device', 'enospc')):
        print('\n推奨：Colabのディスク空き容量を確認してください。')
    else:
        print('\n推奨：ログ末尾を確認。必要なら④を安定版に切り替えて再試行してください。')
    raise RuntimeError(f'InfiniteTalk生成失敗（終了コード {status}）。上に実際のエラーとログ末尾を表示しました。')
RAW_VIDEO = Path(str(RAW_PREFIX) + '.mp4')
assert RAW_VIDEO.exists() and RAW_VIDEO.stat().st_size > 10_000, f'出力動画がありません: {RAW_VIDEO}'
print('✅ 推論完了:', RAW_VIDEO)

## ⑦ 完成MP4の仕上げ・プレビュー・ダウンロード（再推論不要）

⑤で検出した**元の音声長**に合わせて動画を仕上げます。37.46秒の音声なら約37.46秒のMP4を作成。保存済みMP4があれば再利用し、**生成⑥を繰り返しません**。

動画プレビュー失敗は無視してダウンロードへ進み、ダウンロード自体が失敗してもMP4とログを残します。完成した動画はColabを切断する前に保存してください。


In [ ]:
# ⑦ 仕上げ・プレビュー・ダウンロード（v6：再生成なしで安全に再実行可）
from pathlib import Path
from IPython.display import display, HTML, FileLink
from google.colab import files
import subprocess, json, base64, traceback

output_root = Path('/content/infinitetalk_outputs')
# ⑤で作ったジョブを優先。Colabのカーネルを再起動した場合だけ最新の既存ジョブを探す。
if 'JOB' in globals() and Path(JOB).is_dir():
    current_job = Path(JOB)
else:
    available = [p for p in output_root.glob('*') if p.is_dir() and
                 (list(p.glob('InfiniteTalk_*s_*.mp4')) or (p / 'infinitetalk_raw.mp4').is_file())]
    assert available, '生成済み動画が見つかりません。⑥が完了したか確認してください。'
    current_job = max(available, key=lambda p: p.stat().st_mtime)
    print('ℹ️ 最新の生成ジョブを自動選択:', current_job.name)

raw_video = current_job / 'infinitetalk_raw.mp4'
audio_path = current_job / 'speech_16k_mono.wav'

def video_info(path):
    """不完全な出力はNoneで返す。動画ストリームと再生時間を確認。"""
    if not path.is_file() or path.stat().st_size < 1024:
        return None
    try:
        r = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration,size',
             '-show_entries', 'stream=codec_type,codec_name', '-of', 'json', str(path)],
            capture_output=True, text=True, timeout=45, check=True)
        d = json.loads(r.stdout)
        duration = float(d.get('format', {}).get('duration', 0))
        if duration <= 0 or not any(s.get('codec_type') == 'video' for s in d.get('streams', [])):
            return None
        return {'duration': duration, 'size': path.stat().st_size}
    except Exception:
        return None

# 元の秒数をジョブに保存したメタデータから復元（カーネル再起動にも対応）。
meta_path = current_job / 'job_meta.json'
if meta_path.is_file():
    job_meta = json.loads(meta_path.read_text(encoding='utf-8'))
    target_seconds = float(job_meta['target_seconds'])
elif audio_path.is_file():
    # v6生成ジョブ用の互換処理
    audio_probe = subprocess.run(
        ['ffprobe', '-v','error','-show_entries','format=duration',
         '-of','json',str(audio_path)], capture_output=True,text=True,check=True)
    target_seconds = float(json.loads(audio_probe.stdout)['format']['duration'])
else:
    # 出力MP4のみ残っていても既存ファイルを探す
    target_seconds = 0.0

job_name = current_job.name
token = f'{target_seconds:.2f}'.rstrip('0').rstrip('.') if target_seconds else 'saved'
target = current_job / f'InfiniteTalk_{token}s_{job_name}.mp4'
existing = video_info(target)

# 完成MP4があるなら再エンコード不要。壊れている／秒数が違う場合のみ仕上げ直す。
if existing and abs(existing['duration'] - target_seconds) <= 0.5:
    print('✅ 仕上げ済み動画を再利用（FFmpeg処理を省略）:', target)
else:
    # カーネル再起動で秒数設定が復元できなくても、既存の完成MP4があれば優先。
    alternatives = sorted(current_job.glob('InfiniteTalk_*s_*.mp4'),
                          key=lambda p: p.stat().st_mtime, reverse=True)
    valid_alternatives = [(p, video_info(p)) for p in alternatives if p != target]
    valid_alternatives = [(p, info) for p, info in valid_alternatives if info]
    if valid_alternatives and (not raw_video.is_file() or not meta_path.is_file()):
        target, existing = valid_alternatives[0]
        print('✅ 保存済み動画を検出（再生成なし）:', target)
    else:
        assert video_info(raw_video), (
            '完成動画も生成元動画も見つかりません。⑥の処理が正常終了したか確認してください。')
        assert audio_path.is_file(), '仕上げに使う音声がありません: ' + str(audio_path)
        temp_file = current_job / f'.InfiniteTalk_{target_seconds}s_{job_name}.part.mp4'
        try:
            if temp_file.exists():
                temp_file.unlink()
            cmd = [
                'ffmpeg', '-y', '-nostdin', '-hide_banner', '-loglevel', 'error',
                '-i', str(raw_video), '-i', str(audio_path),
                '-filter_complex',
                f'[0:v]fps=25,tpad=stop_mode=clone:stop_duration=3,'
                f'trim=duration={target_seconds},setpts=PTS-STARTPTS[v]',
                '-map', '[v]', '-map', '1:a:0',
                '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
                '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '192k',
                '-t', str(target_seconds), '-movflags', '+faststart', str(temp_file)]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode != 0 or video_info(temp_file) is None:
                raise RuntimeError((result.stderr or 'ffmpegの出力検証に失敗')[-2500:])
            temp_file.replace(target)  # 検証済みファイルだけ正式名にする
            print('✅ 映像と音声の仕上げが完了')
        except Exception as exc:
            print('❌ 仕上げに失敗。元の生成動画は消していません:', raw_video)
            raise RuntimeError('FFmpeg仕上げエラー: ' + str(exc)) from exc

info = video_info(target)
assert info, 'MP4の検証に失敗しました: ' + str(target)
print(f'✅ 完成: {target}')
print(f"長さ: {info['duration']:.2f}秒 / サイズ: {info['size']/1024**2:.1f} MiB")

# リンク・プレビューに失敗してもMP4は維持し、ダウンロードへ進む。
try:
    display(FileLink(str(target)))
except Exception as exc:
    print('ℹ️ ファイルリンクの表示は省略:', str(exc))

try:
    if info['size'] <= 20 * 1024**2:
        b64 = base64.b64encode(target.read_bytes()).decode('ascii')
        display(HTML(
            '<video controls playsinline preload="metadata" width="480" '
            'style="max-width:100%" '
            f'src="data:video/mp4;base64,{b64}"></video>'))
    else:
        print('ℹ️ 大容量MP4のため画面プレビューは省略（動画自体は完成済み）。')
except Exception as exc:
    print('⚠️ プレビューのみ失敗。動画とダウンロードは正常です:', exc)

if globals().get('AUTO_DOWNLOAD', True):
    try:
        files.download(str(target))
        print('✅ ブラウザへのダウンロードをリクエストしました')
    except Exception as exc:
        print('⚠️ 自動ダウンロードを起動できませんでした:', exc)
        print('Colab左の「ファイル」から以下のMP4を右クリック → ダウンロード')
        print(target)
else:
    print('ℹ️ AUTO_DOWNLOAD=False：自動ダウンロードは省略しました。')
print('📂 MP4の保存場所:', target)
print('再生成せずに再取得したい場合は⑦をもう一度実行してください。')


## ⑧ 緊急ダウンロード（完成動画だけ再取得）

Colabのカーネルが再起動しても `/content/infinitetalk_outputs` が残っていれば、生成⑥なしで最新の完成動画をダウンロードできます。


In [ ]:
# ⑧ 緊急用：最新の完成MP4を再ダウンロード（⑥・⑦の再実行は不要）
from pathlib import Path
from google.colab import files
import subprocess

root = Path('/content/infinitetalk_outputs')
completed = sorted(root.glob('*/InfiniteTalk_*s_*.mp4'),
                   key=lambda p: p.stat().st_mtime, reverse=True)
if not completed:
    print('完成したMP4がありません。Colab左側「ファイル」からジョブフォルダも確認してください。')
else:
    latest = completed[0]
    print('📂 保存先:', latest)
    print(f'サイズ: {latest.stat().st_size/1024**2:.1f} MiB')
    try:
        files.download(str(latest))
    except Exception as e:
        print('ダウンロードが起動しませんでした。左の「ファイル」から直接保存してください:', e)


---
### 高速化設定と注意事項

- **G4高速**：`--offload_model False`。オフロードの転送を削減する試みです。実測の高速化率は未検証
- **安定版**：前回40秒の動画生成に成功した `--offload_model True` の構成
- **省VRAM**：`--offload_model True` と `--t5_cpu`, `--num_persistent_param_in_dit 0`（低速）
- **TeaCache**：設定④の試験スイッチ。8ステップ量子化モデルへの品質・速度影響は未検証
- **音声長**：自動。`TEST_FIRST_SECONDS=0` は全長。例：65.18秒なら目標65.18秒
- **短い音声**：4秒未満だけモデル内部で無音を追加し、仕上げ時に元の音声の長さへ戻します
- **生成目安**：最初81フレーム、次から72フレームずつ。G4高速モードの実速度は実際に生成して確認してください
- **FlashAttention 2不要**：前回成功したSDPAパッチを継続。環境のインストール・依存関係はv6と同じ
- **Colab制限**：長尺は数時間〜数十時間かかる場合があります。長すぎる音声はランタイム切断のリスク

**初回の試し方：** ④「G4高速・先頭5秒・TeaCache OFF」→⑤→⑥→⑦。成功と時間を確認してから④の上限を0にし、⑤で再アップロードしてください。再アップロードで新しいジョブが作られます。

失敗時は⑥の末尾にログを表示します。`/content/infinitetalk_outputs/<job>/generate.log` を保存してください。
